# Kassow KR810 -- Tier 0-4 Kinematic Evaluation (Kaggle Template)

This notebook runs the **Tier 0-4 evaluation pipeline** for the Kassow KR810 (7-DOF
collaborative manipulator) and displays the results. It is a *runner*, not a re-implementation:
every algorithm (forward kinematics, Jacobian, Damped Least Squares, trajectory/feasibility
metrics) lives in the repository's `kinematics/`, `algorithms/`, and `evaluation/` packages and
is invoked through `pipelines.run_tier0_to_tier4`, never copied into this notebook.

## Scope

- **Trajectory-first, kinematic evaluation only**: Tier 0 (kinematics validation) through
  Tier 4 (joint feasibility/smoothness). No actuator dynamics, no torque-level control.
- **No PPO, no MPDIK, no MAPPO.** This notebook produces evaluation data that a later stage may
  use to motivate an MPDIK design; it does not implement or train any controller/policy here.
- `ee_site` is the MuJoCo **end-effector link reference frame** defined in `assets/kr810.xml`
  (see `configs/robot_config.json`). It is **not** a calibrated tool-center-point (TCP); no
  physical TCP calibration has been performed.
- The position/orientation thresholds used below (e.g. 4/6/10 mm, 10 degrees) are
  **project-defined acceptance criteria** for this dataset's own pipeline
  (`configs/evaluation_config.json`). They are **not** an ISO 9283 certification, and no ISO
  9283 certification is claimed anywhere in this notebook or its outputs.

## What this notebook does

1. Locates the project (Kaggle input Dataset, or the local repository).
2. Copies it into a writable working directory (Kaggle input is read-only).
3. Checks/installs the pinned Python dependencies.
4. Validates the dataset (model, benchmark, trajectories) before running anything.
5. Runs `python -m pipelines.run_tier0_to_tier4` with the configuration from Cell 2.
6. Loads and displays `FINAL_SUMMARY.json` and the per-tier CSV outputs.
7. Packages the run's results (not the source dataset) into a ZIP file.


In [ ]:
# ============================================================
# USER CONFIGURATION -- edit the values below, then Run All.
# ============================================================

PRESET = "smoke"  # "smoke" (fast sanity check) or "full" (entire dataset, CPU/time intensive)
SEED = 42

METHODS = ["warm_start", "cold_start"]  # subset of {"warm_start", "cold_start"}

TRAJECTORY_IDS = None      # e.g. ["line_fixed_orientation", "circle_fixed_orientation"]; None = preset default
TRIAL_CATEGORY = "all"     # "repeatability" | "robustness" | "all"

TRIAL_LIMIT = None         # int > 0, or None for preset default
POINT_SAMPLE_LIMIT = None  # int > 0, or None for preset default
WAYPOINT_LIMIT = None      # int > 0, or None for preset default

RUN_PLOTS = True
RESUME = False
OVERWRITE = True

RUN_NAME = "kr810_tier0_tier4"

# Informational only: these mirror configs/evaluation_config.json's defaults. The Tier 0-4 CLI
# has no flag to override them -- they are validated here and used only for display/comparison
# in the result tables below (Cell 13, Cell 15), never forwarded to the pipeline invocation.
POSITION_THRESHOLD_M = 0.006
ORIENTATION_THRESHOLD_DEG = 10.0

RUN_PRECHECK_TESTS = True  # used by Cell 10

# ------------------------------------------------------------
# Validation -- do not edit below this line.
# ------------------------------------------------------------
_VALID_PRESETS = {"smoke", "full"}
_VALID_METHODS = {"warm_start", "cold_start"}
_VALID_TRIAL_CATEGORIES = {"repeatability", "robustness", "all"}


def _validate_user_config():
    errors = []
    if PRESET not in _VALID_PRESETS:
        errors.append(f"PRESET must be one of {_VALID_PRESETS}, got {PRESET!r}")
    if not METHODS or not set(METHODS).issubset(_VALID_METHODS):
        errors.append(f"METHODS must be a non-empty subset of {_VALID_METHODS}, got {METHODS!r}")
    if TRIAL_CATEGORY not in _VALID_TRIAL_CATEGORIES:
        errors.append(f"TRIAL_CATEGORY must be one of {_VALID_TRIAL_CATEGORIES}, got {TRIAL_CATEGORY!r}")
    for name, value in (
        ("TRIAL_LIMIT", TRIAL_LIMIT),
        ("POINT_SAMPLE_LIMIT", POINT_SAMPLE_LIMIT),
        ("WAYPOINT_LIMIT", WAYPOINT_LIMIT),
    ):
        if value is not None and value <= 0:
            errors.append(f"{name} must be > 0 when set, got {value!r}")
    if POSITION_THRESHOLD_M <= 0:
        errors.append(f"POSITION_THRESHOLD_M must be > 0, got {POSITION_THRESHOLD_M!r}")
    if ORIENTATION_THRESHOLD_DEG <= 0:
        errors.append(f"ORIENTATION_THRESHOLD_DEG must be > 0, got {ORIENTATION_THRESHOLD_DEG!r}")
    if RESUME and OVERWRITE:
        errors.append("RESUME and OVERWRITE cannot both be True")
    if errors:
        raise ValueError("Invalid user configuration:\n- " + "\n- ".join(errors))
    print("User configuration OK.")


_validate_user_config()


In [ ]:
# ---- Environment diagnostics ----
import os
import platform
import shutil
import sys
from pathlib import Path

IN_KAGGLE = (
    Path("/kaggle/input").is_dir()
    or "KAGGLE_KERNEL_RUN_TYPE" in os.environ
    or "KAGGLE_URL_BASE" in os.environ
)

print(f"Python version    : {sys.version.split()[0]}")
print(f"Platform          : {platform.platform()}")
print(f"Working directory : {Path.cwd()}")
print(f"Kaggle detected   : {IN_KAGGLE}")
print(f"CPU count         : {os.cpu_count()}")

try:
    import numpy

    print(f"NumPy version     : {numpy.__version__}")
except ImportError:
    print("NumPy version     : not installed yet (see Cell 6)")

try:
    import mujoco

    print(f"MuJoCo version    : {mujoco.__version__}")
except ImportError:
    print("MuJoCo version    : not installed yet (see Cell 6)")

gpu_tool = shutil.which("nvidia-smi")
print(f"nvidia-smi found  : {gpu_tool is not None} (informational only)")
print("Note: the Tier 0-4 evaluation pipeline runs entirely on CPU; a GPU is not required.")


In [ ]:
# ---- Locate dataset ----
import json

# The project's own manifest identifier (DATASET_MANIFEST.json's "dataset_name" field) -- this
# is an internal project identifier, not a Kaggle Dataset slug/owner path.
EXPECTED_DATASET_NAME = "kassow-kr810-trajectory-tier0-tier4"

# Set this to bypass auto-discovery (e.g. multiple ambiguous /kaggle/input candidates, or a
# non-standard local checkout layout). Leave as None for normal auto-discovery.
DATASET_ROOT_OVERRIDE = None


def _looks_like_project_root(path: Path) -> bool:
    manifest_path = path / "DATASET_MANIFEST.json"
    if not manifest_path.is_file():
        return False
    try:
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    except (json.JSONDecodeError, OSError):
        return False
    if manifest.get("dataset_name") != EXPECTED_DATASET_NAME:
        return False
    if not (path / "assets" / "kr810.xml").is_file():
        return False
    if not (path / "pipelines" / "run_tier0_to_tier4.py").is_file():
        return False
    return True


if DATASET_ROOT_OVERRIDE is not None:
    PROJECT_ROOT = Path(DATASET_ROOT_OVERRIDE)
    if not _looks_like_project_root(PROJECT_ROOT):
        raise FileNotFoundError(
            f"DATASET_ROOT_OVERRIDE={PROJECT_ROOT} does not look like the project root "
            "(missing DATASET_MANIFEST.json, assets/kr810.xml, or pipelines/run_tier0_to_tier4.py)."
        )
elif IN_KAGGLE:
    kaggle_input = Path("/kaggle/input")
    candidates = []
    if kaggle_input.is_dir():
        for child in sorted(p for p in kaggle_input.iterdir() if p.is_dir()):
            if _looks_like_project_root(child):
                manifest = json.loads((child / "DATASET_MANIFEST.json").read_text(encoding="utf-8"))
                candidates.append((child, str(manifest.get("version", "0.0.0"))))

    if len(candidates) == 0:
        raise FileNotFoundError(
            "No candidate dataset found under /kaggle/input matching "
            f"dataset_name={EXPECTED_DATASET_NAME!r}. Attach the project Dataset to this "
            "notebook (Add Input) and re-run, or set DATASET_ROOT_OVERRIDE above."
        )
    elif len(candidates) == 1:
        PROJECT_ROOT = candidates[0][0]
    else:
        def _version_key(v):
            try:
                return tuple(int(part) for part in v.split("."))
            except ValueError:
                return None

        keyed = [(c, _version_key(v)) for c, v in candidates]
        if any(k is None for _, k in keyed) or len({k for _, k in keyed}) != len(keyed):
            listing = "\n".join(f"  - {c} (version={v})" for c, v in candidates)
            raise RuntimeError(
                "Multiple candidate datasets found under /kaggle/input and their versions "
                "cannot be safely ranked:\n" + listing +
                "\nSet DATASET_ROOT_OVERRIDE above to the correct path."
            )
        PROJECT_ROOT = max(keyed, key=lambda item: item[1])[0]
else:
    PROJECT_ROOT = None
    for base in [Path.cwd(), *Path.cwd().parents]:
        if _looks_like_project_root(base):
            PROJECT_ROOT = base
            break
    if PROJECT_ROOT is None:
        raise FileNotFoundError(
            "Could not locate the project root (DATASET_MANIFEST.json + assets/kr810.xml + "
            "pipelines/run_tier0_to_tier4.py) from the current directory or its parents. Run "
            "this notebook from within the repository, or set DATASET_ROOT_OVERRIDE above."
        )

print(f"PROJECT_ROOT resolved to: {PROJECT_ROOT}")


In [ ]:
# ---- Copy project to working directory (Kaggle input is read-only) ----
_EXCLUDE_DIR_NAMES = {
    ".git", ".venv", "venv", "__pycache__", ".pytest_cache", ".ipynb_checkpoints",
    "results", "temporary_results",
}
_EXCLUDE_FILE_SUFFIXES = {".zip"}


def _ignore_for_copy(directory, names):
    ignored = set()
    for name in names:
        full = Path(directory) / name
        if full.is_dir() and name in _EXCLUDE_DIR_NAMES:
            ignored.add(name)
        elif full.is_file() and full.suffix in _EXCLUDE_FILE_SUFFIXES:
            ignored.add(name)
    return ignored


if IN_KAGGLE:
    WORK_ROOT = Path("/kaggle/working/Kassow_KR810_Trajectory_Tier0_Tier4")

    if WORK_ROOT.exists():
        if OVERWRITE:
            if WORK_ROOT.resolve() in (Path("/kaggle/working").resolve(), WORK_ROOT.resolve().parent):
                raise RuntimeError(f"refusing to remove an unsafe working path: {WORK_ROOT}")
            shutil.rmtree(WORK_ROOT)
            shutil.copytree(PROJECT_ROOT, WORK_ROOT, ignore=_ignore_for_copy)
        elif RESUME:
            if not _looks_like_project_root(WORK_ROOT):
                raise FileNotFoundError(
                    f"RESUME=True but no valid working copy exists at {WORK_ROOT}; "
                    "set OVERWRITE=True for a first run."
                )
        else:
            if not _looks_like_project_root(WORK_ROOT):
                raise FileExistsError(
                    f"{WORK_ROOT} exists but is not a valid project copy, and neither OVERWRITE "
                    "nor RESUME is set. Enable one of them in Cell 2."
                )
    else:
        shutil.copytree(PROJECT_ROOT, WORK_ROOT, ignore=_ignore_for_copy)
else:
    # Outside Kaggle there is no read-only input mount to work around: run directly against
    # the local project root (output still lands under a separate results/ directory, Cell 7).
    WORK_ROOT = PROJECT_ROOT

print(f"WORK_ROOT resolved to: {WORK_ROOT}")


In [ ]:
# ---- Dependency check ----
import importlib
import subprocess

_REQUIRED_MODULES = ["numpy", "scipy", "pandas", "matplotlib", "mujoco", "tqdm", "jsonschema", "nbformat"]


def _missing_modules(modules):
    missing = []
    for name in modules:
        try:
            importlib.import_module(name)
        except ImportError:
            missing.append(name)
    return missing


missing = _missing_modules(_REQUIRED_MODULES)
if missing:
    print(f"Missing packages: {missing}. Installing from requirements.txt ...")
    requirements_path = WORK_ROOT / "requirements.txt"
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", str(requirements_path)],
        capture_output=True, text=True,
    )
    print(result.stdout[-4000:])
    if result.returncode != 0:
        print(result.stderr[-4000:], file=sys.stderr)
        raise RuntimeError(
            f"Dependency installation failed (returncode {result.returncode}). If this is a "
            "Kaggle notebook with internet access disabled, enable it under Notebook Settings "
            "(Internet: On), or use a Kaggle environment/Dataset with these packages preinstalled."
        )
    missing = _missing_modules(_REQUIRED_MODULES)
    if missing:
        raise RuntimeError(f"Still missing after installation attempt: {missing}")

print("Dependency versions:")
for name in _REQUIRED_MODULES:
    module = importlib.import_module(name)
    print(f"  {name:<12} {getattr(module, '__version__', 'unknown')}")


In [ ]:
# ---- Import path and working directory ----
if str(WORK_ROOT) not in sys.path:
    sys.path.insert(0, str(WORK_ROOT))
os.chdir(WORK_ROOT)

import algorithms  # noqa: F401 -- import-only check, confirms WORK_ROOT is importable
import evaluation  # noqa: F401
import kinematics  # noqa: F401
import pipelines  # noqa: F401

if IN_KAGGLE:
    OUTPUT_ROOT = Path("/kaggle/working/kr810_results") / RUN_NAME
else:
    OUTPUT_ROOT = WORK_ROOT / "results" / RUN_NAME

print(f"WORK_ROOT   : {WORK_ROOT}")
print(f"OUTPUT_ROOT : {OUTPUT_ROOT}")


In [ ]:
# ---- Validate dataset ----
import pandas as pd

from kinematics.model_loader import load_model_context
from utils.file_checksum import sha256_file

_validation_rows = []


def _check(name, condition, detail=""):
    _validation_rows.append({"check": name, "passed": bool(condition), "detail": detail})
    return condition


manifest_path = WORK_ROOT / "DATASET_MANIFEST.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
_check("DATASET_MANIFEST.json parses", True, f"version={manifest.get('version')}")
_check("benchmark_status == generated_and_validated", manifest.get("benchmark_status") == "generated_and_validated")
_check("trajectories_status == generated_and_validated", manifest.get("trajectories_status") == "generated_and_validated")
_check("assets/kr810.xml exists", (WORK_ROOT / "assets" / "kr810.xml").is_file())

point_ik_path = WORK_ROOT / "benchmarks" / "point_ik" / "point_ik_v1.npz"
_check("benchmarks/point_ik/point_ik_v1.npz exists", point_ik_path.is_file())

point_ik_checksum_path = WORK_ROOT / "benchmarks" / "point_ik" / "point_ik_checksum.json"
if point_ik_checksum_path.is_file() and point_ik_path.is_file():
    checksum_manifest = json.loads(point_ik_checksum_path.read_text(encoding="utf-8"))
    expected_sha256 = next(
        (f["sha256"] for f in checksum_manifest.get("files", []) if f["filename"].endswith("point_ik_v1.npz")),
        None,
    )
    actual_sha256 = sha256_file(point_ik_path)
    _check(
        "point_ik_v1.npz sha256 matches checksum manifest",
        expected_sha256 is not None and expected_sha256 == actual_sha256,
    )

trajectory_manifest_path = WORK_ROOT / "trajectories" / "trajectory_manifest.csv"
trajectory_trials_path = WORK_ROOT / "trajectories" / "trajectory_trials.csv"
_check("trajectories/trajectory_manifest.csv exists", trajectory_manifest_path.is_file())
_check("trajectories/trajectory_trials.csv exists", trajectory_trials_path.is_file())

if trajectory_manifest_path.is_file():
    trajectory_manifest_df = pd.read_csv(trajectory_manifest_path)
    _check(
        "8 trajectory rows in trajectory_manifest.csv",
        len(trajectory_manifest_df) == 8,
        f"found {len(trajectory_manifest_df)}",
    )
    missing_files = [
        row["file_path"] for _, row in trajectory_manifest_df.iterrows()
        if not (WORK_ROOT / row["file_path"]).is_file()
    ]
    _check("all 8 trajectory NPZ files exist", len(missing_files) == 0, f"missing={missing_files}")
    _check(
        "all trajectory rows generation_status == validated",
        bool((trajectory_manifest_df["generation_status"] == "validated").all()),
    )

model_context = load_model_context()
_check("model nq == 7", model_context.nq == 7, f"nq={model_context.nq}")
_check("model nv == 7", model_context.nv == 7, f"nv={model_context.nv}")
_check("ee_site resolved in compiled model", model_context.ee_site_name == "ee_site" and model_context.ee_site_id >= 0)

model_sha256 = sha256_file(WORK_ROOT / "assets" / "kr810.xml")
print(f"assets/kr810.xml sha256 (informational, no fixed reference published): {model_sha256}\n")

validation_df = pd.DataFrame(_validation_rows)
print(validation_df.to_string(index=False))

if not validation_df["passed"].all():
    failed = validation_df[~validation_df["passed"]]
    raise RuntimeError(f"Dataset validation failed:\n{failed.to_string(index=False)}")

print("\nDataset validation: all checks passed.")


In [ ]:
# ---- Resolve pipeline arguments ----
argv = [
    "--preset", PRESET,
    "--output", str(OUTPUT_ROOT),
    "--seed", str(SEED),
    "--methods", ",".join(METHODS),
    "--trial-category", TRIAL_CATEGORY,
    "--log-level", "INFO",
]

if TRAJECTORY_IDS is not None:
    argv += ["--trajectory-ids", ",".join(TRAJECTORY_IDS)]
if TRIAL_LIMIT is not None:
    argv += ["--trial-limit", str(TRIAL_LIMIT)]
if POINT_SAMPLE_LIMIT is not None:
    argv += ["--point-sample-limit", str(POINT_SAMPLE_LIMIT)]
if WAYPOINT_LIMIT is not None:
    argv += ["--waypoint-limit", str(WAYPOINT_LIMIT)]
if not RUN_PLOTS:
    argv += ["--no-plots"]
if OVERWRITE:
    argv += ["--overwrite"]
elif RESUME:
    argv += ["--resume"]

resolved_run_config = {
    "preset": PRESET,
    "seed": SEED,
    "methods": METHODS,
    "trajectory_ids": TRAJECTORY_IDS or "preset default",
    "trial_category": TRIAL_CATEGORY,
    "trial_limit": TRIAL_LIMIT if TRIAL_LIMIT is not None else "preset default",
    "point_sample_limit": POINT_SAMPLE_LIMIT if POINT_SAMPLE_LIMIT is not None else "preset default",
    "waypoint_limit": WAYPOINT_LIMIT if WAYPOINT_LIMIT is not None else "preset default",
    "plots": RUN_PLOTS,
    "overwrite": OVERWRITE,
    "resume": RESUME,
    "output_dir_name": OUTPUT_ROOT.name,
}

print("Resolved run configuration:")
for key, value in resolved_run_config.items():
    print(f"  {key:<18}: {value}")

print("\nEquivalent CLI invocation:")
print("python -m pipelines.run_tier0_to_tier4 " + " ".join(argv))


In [ ]:
if RUN_PRECHECK_TESTS:
    precheck_tests = [
        "tests/test_asset_loading.py",
        "tests/test_model_dimensions.py",
        "tests/test_forward_kinematics.py",
        "tests/test_jacobian.py",
    ]
    result = subprocess.run(
        [sys.executable, "-m", "pytest", *precheck_tests, "-q"],
        cwd=str(WORK_ROOT), capture_output=True, text=True,
    )
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr, file=sys.stderr)
        raise RuntimeError("Precheck tests failed; see pytest output above.")
    print("Precheck tests passed.")
else:
    print("RUN_PRECHECK_TESTS is False; skipping precheck tests.")


In [ ]:
# ---- Run pipeline ----
import time

print("Running: python -m pipelines.run_tier0_to_tier4 " + " ".join(argv))
_start_perf = time.perf_counter()
result = subprocess.run(
    [sys.executable, "-m", "pipelines.run_tier0_to_tier4", *argv],
    cwd=str(WORK_ROOT),
)
elapsed_s = time.perf_counter() - _start_perf

print(f"\nPipeline exit code : {result.returncode}")
print(f"Pipeline runtime   : {elapsed_s:.1f} s")

if result.returncode != 0:
    raise RuntimeError(f"pipelines.run_tier0_to_tier4 failed with exit code {result.returncode}")

final_summary_path = OUTPUT_ROOT / "FINAL_SUMMARY.json"
if not final_summary_path.is_file():
    raise FileNotFoundError(f"expected {final_summary_path} to exist after a successful run")

print(f"FINAL_SUMMARY.json created at: {final_summary_path}")


In [ ]:
final_summary = json.loads((OUTPUT_ROOT / "FINAL_SUMMARY.json").read_text(encoding="utf-8"))
run_manifest = json.loads((OUTPUT_ROOT / "run_manifest.json").read_text(encoding="utf-8"))
resolved_config = json.loads((OUTPUT_ROOT / "resolved_config.json").read_text(encoding="utf-8"))

print(f"Run ID         : {final_summary['run_id']}")
print(f"Preset         : {final_summary['preset']}")
print(f"Overall status : {final_summary['overall_status']}")
print(f"Runtime (s)    : {run_manifest['duration_s']:.1f}")
print(f"Tier 0 gate    : {final_summary['tier0']['status']}")
print(f"Tier 1         : {final_summary['tier1']['status']}")
print(f"Tier 2         : {final_summary['tier2']['status']}")
print(f"Tier 3         : {final_summary['tier3']['status']}")
print(f"Tier 4         : {final_summary['tier4']['status']}")

if final_summary["warnings"]:
    print("\nWarnings:")
    for warning in final_summary["warnings"]:
        print(f"  - {warning}")
if final_summary["fatal_error"]:
    print(f"\nFatal error: {final_summary['fatal_error']}")

print(
    "\nNote: 'Tier execution completed' means each tier ran and produced valid output -- it "
    "does NOT mean every acceptance_criterion below passed. See the acceptance_criteria list "
    "and the per-tier status fields in FINAL_SUMMARY.json for that."
)


In [ ]:
tier0 = final_summary["tier0"]
tier0_table = pd.DataFrame([{
    "gate_pass": tier0.get("gate_pass"),
    "max_jacobian_relative_error": tier0.get("max_jacobian_relative_error"),
    "minimum_sigma_min": tier0.get("minimum_sigma_min"),
    "sample_count": tier0.get("sample_count"),
}])
print("Tier 0 -- kinematics validation gate")
print(tier0_table.to_string(index=False))

tier1 = final_summary["tier1"]
tier1_ci = tier1.get("success_rate_wilson_ci") or {}
tier1_table = pd.DataFrame([{
    "sample_count": tier1.get("sample_count"),
    "success_rate": tier1.get("success_rate"),
    "success_rate_ci_lower": tier1_ci.get("lower"),
    "success_rate_ci_upper": tier1_ci.get("upper"),
    "position_rmse_mm": tier1.get("position_rmse_mm"),
    "position_p95_mm": tier1.get("position_p95_mm"),
    "position_max_mm": tier1.get("position_max_mm"),
    "orientation_rmse_deg": tier1.get("orientation_rmse_deg"),
    "orientation_p95_deg": tier1.get("orientation_p95_deg"),
    "mean_iterations": tier1.get("mean_iterations"),
    "p95_runtime_ms": tier1.get("p95_runtime_ms"),
    "acceptance_status": tier1.get("status"),
}])
print("\nTier 1 -- point DLS (project acceptance thresholds: "
      f"position_threshold={POSITION_THRESHOLD_M * 1000:.1f} mm, "
      f"orientation_threshold={ORIENTATION_THRESHOLD_DEG:.1f} deg)")
print(tier1_table.to_string(index=False))


In [ ]:
warm_vs_cold_path = OUTPUT_ROOT / "tier2_sequential_dls" / "warm_vs_cold.csv"
trial_summaries_path = OUTPUT_ROOT / "tier2_sequential_dls" / "trajectory_trial_summaries.csv"

if warm_vs_cold_path.is_file() and warm_vs_cold_path.stat().st_size > 0:
    warm_vs_cold_df = pd.read_csv(warm_vs_cold_path)
else:
    warm_vs_cold_df = pd.DataFrame()

if not warm_vs_cold_df.empty:
    print("Warm-start vs cold-start (per trial):")
    print(warm_vs_cold_df.to_string(index=False))
else:
    print("warm_vs_cold.csv has no rows for this selection.")

if trial_summaries_path.is_file():
    trial_summaries_df = pd.read_csv(trial_summaries_path)
else:
    trial_summaries_df = pd.DataFrame()

if not trial_summaries_df.empty:
    summary_by_method = trial_summaries_df.groupby("method").agg(
        waypoint_success_rate=("waypoint_success_rate", "mean"),
        full_trajectory_completion_rate=("full_trajectory_completed", "mean"),
        mean_iterations=("mean_iterations", "mean"),
        p95_iterations=("p95_iterations", "mean"),
        mean_solve_time_ms=("mean_solve_time_ms", "mean"),
        p95_solve_time_ms=("p95_solve_time_ms", "mean"),
        maximum_failure_streak=("maximum_failure_streak", "max"),
        recovery_rate=("recovery_rate", "mean"),
    )
    print("\nMethod comparison (aggregated over selected trials):")
    print(summary_by_method.to_string())
    print(
        "\nNote: compare the numbers above directly -- do not assume warm-start is always "
        "better than cold-start without checking success/completion rate and iteration/runtime "
        "cost for this specific selection."
    )
else:
    print("\ntrajectory_trial_summaries.csv has no rows for this selection.")


In [ ]:
tier3_dir = OUTPUT_ROOT / "tier3_trajectory_tracking"
tracking_df = pd.read_csv(tier3_dir / "trajectory_metrics.csv")
cross_track_df = pd.read_csv(tier3_dir / "cross_track_metrics.csv")

iso_path = tier3_dir / "iso9283_metrics.csv"
iso_df = pd.read_csv(iso_path) if iso_path.is_file() and iso_path.stat().st_size > 0 else pd.DataFrame()

print("Tier 3 -- position/orientation tracking, per (trial_id, method):")
tracking_cols = [
    "trial_id", "trajectory_id", "method",
    "position_rmse_m", "position_p95_m", "position_max_m",
    "orientation_rmse_deg", "orientation_p95_deg",
]
print(tracking_df[tracking_cols].to_string(index=False))

if not cross_track_df.empty:
    print("\nCross-track metrics, per (trial_id, method):")
    cross_cols = [
        "trial_id", "trajectory_id", "method",
        "cross_track_rmse_m", "cross_track_p95_m", "final_progress_ratio",
    ]
    print(cross_track_df[cross_cols].to_string(index=False))

if not iso_df.empty:
    print("\nISO 9283-inspired accuracy/repeatability (ATp/RTp) -- inspired by ISO 9283, "
          "NOT a certified ISO 9283 result:")
    iso_cols = ["trajectory_id", "method", "speed_scale", "n_repeats", "atp_m", "rtp_m", "warning"]
    print(iso_df[iso_cols].to_string(index=False))
else:
    print(
        "\nISO 9283-inspired ATp/RTp metrics: unavailable for this selection (no "
        "(trajectory_id, method, speed_scale) group had >= 2 repeatability repeats). Reported "
        "as unavailable -- not shown as 0."
    )


In [ ]:
tier4_dir = OUTPUT_ROOT / "tier4_joint_feasibility"
smoothness_df = pd.read_csv(tier4_dir / "smoothness_metrics.csv")
feasibility_df = pd.read_csv(tier4_dir / "joint_feasibility_metrics.csv")
singularity_df = pd.read_csv(tier4_dir / "singularity_path_metrics.csv")
runtime_df = pd.read_csv(tier4_dir / "runtime_metrics.csv")

for method in METHODS:
    m_smooth = smoothness_df[smoothness_df["method"] == method]
    m_feas = feasibility_df[feasibility_df["method"] == method]
    m_sing = singularity_df[singularity_df["method"] == method]
    m_runtime = runtime_df[runtime_df["method"] == method]
    if m_smooth.empty:
        continue

    print(f"\nTier 4 -- {method}:")
    print(f"  max_joint_jump_rad      : {m_smooth['max_joint_jump_rad'].max():.6f}")

    rms_jerk = m_smooth["global_rms_jerk_rad_s3"].dropna()
    print(f"  rms_jerk_rad_s3         : {rms_jerk.mean():.6f}" if not rms_jerk.empty
          else "  rms_jerk_rad_s3         : unavailable")

    margin = m_feas["minimum_normalized_joint_limit_margin"].dropna() if not m_feas.empty else pd.Series(dtype=float)
    print(f"  min_joint_limit_margin  : {margin.min():.6f}" if not margin.empty
          else "  min_joint_limit_margin  : unavailable")

    if not m_feas.empty:
        violation_rate = float((m_feas["velocity_violation_count"].fillna(0) > 0).mean())
        print(f"  velocity_violation_rate : {violation_rate:.4f}")
    else:
        print("  velocity_violation_rate : unavailable")

    print(f"  minimum_sigma_min       : {m_sing['minimum_sigma_min'].min():.6f}" if not m_sing.empty
          else "  minimum_sigma_min       : unavailable")
    print(f"  near_singular_fraction  : {m_sing['near_singular_fraction'].mean():.4f}" if not m_sing.empty
          else "  near_singular_fraction  : unavailable")
    print(f"  deadline_miss_rate      : {m_runtime['deadline_miss_rate'].mean():.4f}" if not m_runtime.empty
          else "  deadline_miss_rate      : unavailable")

    accel_statuses = set(m_feas["acceleration_status"].unique()) if not m_feas.empty else set()
    accel_status = "available" if accel_statuses == {"available"} else "unavailable"
    print(f"  acceleration_status     : {accel_status} "
          "(no acceleration limits configured -- reported as unavailable, not a fabricated pass)")


In [ ]:
MAX_FIGURES_TO_DISPLAY = 6

_PRIORITY_EXACT_NAMES = ["position_error_cdf.png", "warm_vs_cold_success.png"]
_PRIORITY_PREFIXES = ["target_vs_actual", "position_error_", "joint_trajectory_", "sigma_min_"]

if RUN_PLOTS:
    all_figures = sorted(OUTPUT_ROOT.rglob("figures/*.png"))
    if not all_figures:
        print("RUN_PLOTS is True but no figure PNGs were found for this selection.")
    else:
        from IPython.display import Image, display

        def _priority(path):
            if path.name in _PRIORITY_EXACT_NAMES:
                return 0
            for rank, prefix in enumerate(_PRIORITY_PREFIXES, start=1):
                if path.name.startswith(prefix):
                    return rank
            return len(_PRIORITY_PREFIXES) + 1

        selected = sorted(all_figures, key=_priority)[:MAX_FIGURES_TO_DISPLAY]
        print(f"Displaying {len(selected)} of {len(all_figures)} generated figures:")
        for figure_path in selected:
            print(f"\n{figure_path.relative_to(OUTPUT_ROOT)}")
            display(Image(filename=str(figure_path)))
else:
    print("RUN_PLOTS is False; plotting was disabled for this run (--no-plots).")


In [ ]:
# ---- Package results ----
import hashlib
import zipfile

if IN_KAGGLE:
    zip_path = Path("/kaggle/working") / f"KR810_Tier0_Tier4_{RUN_NAME}.zip"
else:
    zip_path = OUTPUT_ROOT.parent / f"KR810_Tier0_Tier4_{RUN_NAME}.zip"

if zip_path.exists():
    zip_path.unlink()

file_count = 0
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for file_path in sorted(OUTPUT_ROOT.rglob("*")):
        if file_path.is_file():
            zf.write(file_path, arcname=file_path.relative_to(OUTPUT_ROOT))
            file_count += 1

zip_size_bytes = zip_path.stat().st_size
zip_sha256 = hashlib.sha256(zip_path.read_bytes()).hexdigest()

print(f"ZIP path : {zip_path}")
print(f"ZIP size : {zip_size_bytes / 1e6:.2f} MB")
print(f"Files    : {file_count}")
print(f"SHA256   : {zip_sha256}")


## Final research note

- This is a **Tier 0-4 kinematic evaluation** result: forward kinematics, Jacobian, and DLS
  inverse kinematics accuracy/feasibility, evaluated against the KR810 MuJoCo model.
- There is **no actuator dynamics and no controller/tracking-loop** in this pipeline -- Tier 3's
  "tracking error" is the error between the commanded IK joint solution and the intended
  Cartesian trajectory, not a simulated servo response.
- There is **no MPDIK** (or any learned/optimization-based IK) in this pipeline; only the DLS
  variants in `algorithms/` (point, sequential warm-start, sequential cold-start).
- The position/orientation errors reported above are **IK/kinematic-trajectory errors**, not
  measured physical robot errors.
- This Tier 0-4 dataset exists to characterize where DLS struggles (large steps, near-singular
  configurations, orientation-heavy waypoints) so that a later stage can motivate and evaluate
  MPDIK against a concrete, quantified baseline.
- The thresholds referenced throughout this notebook (`configs/evaluation_config.json`) are
  **project-defined acceptance criteria**, not an ISO 9283 certification, and no ISO 9283
  certification is claimed.
